# SensiFake: Qwen3-VL Gold-development pilot

Zero-shot evaluation of Qwen/Qwen3-VL-4B-Instruct as a semantic-sensitivity annotator on the 101 frozen human Gold-development images. This notebook does not perform deepfake detection and does not process the full 3,000-image corpus.

## 1. Imports

Install a Transformers release with native Qwen3-VL support, then import the PyTorch inference and evaluation stack. Kaggle Internet must be enabled for package and model access.

In [ ]:
%pip install -q -U "transformers>=4.57.0" accelerate huggingface_hub

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import random
import re
import time
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import huggingface_hub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import transformers
from huggingface_hub import HfApi
from IPython.display import display
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
)
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, set_seed

## 2. Globals

All input roots are explicit. In particular, the notebook never recursively searches sensifake600, whose separate legacy 600-image tree must remain excluded.

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION_REQUESTED = "main"
EXPECTED_GOLD_SHA256 = "c783a28509ca200d37d7b9e0f79a4bdd359c3f59f16acf2a0a80759875eb573f"
EXPECTED_GOLD_ROWS = 101
EXPECTED_MANIFEST_ROWS = 600

SOURCE_DATASET_ROOT = Path("/kaggle/input/sensifake600")
PILOT_ROOT = Path("/kaggle/input/sensifake600/datasets/datasets/openfake/pilot-600")
MANIFEST_PATH = PILOT_ROOT / "manifest.jsonl"
LEGACY_ROOT = SOURCE_DATASET_ROOT / "sensifake-600"
GOLD_CSV_PATH = Path("/kaggle/input/sensifake-gold-development/sensitivity_annotations.csv")

OUTPUT_DIR = Path("/kaggle/working/qwen3vl_gold_pilot")
PREDICTIONS_PATH = OUTPUT_DIR / "qwen3vl_gold_predictions.csv"
METRICS_PATH = OUTPUT_DIR / "qwen3vl_gold_metrics.json"
DISAGREEMENTS_PATH = OUTPUT_DIR / "qwen3vl_gold_disagreements.csv"
RUN_METADATA_PATH = OUTPUT_DIR / "qwen3vl_run_metadata.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGE_SIDE = 1280
GENERATION_PARAMETERS = {
    "max_new_tokens": 220,
    "do_sample": False,
    "use_cache": True,
}
LEVEL_ORDER = ["low", "medium", "high"]

GOLD_REQUIRED_COLUMNS = (
    "content_hash", "blind_id", "public_relevance", "harm_urgency",
    "vulnerability", "sensitivity_score", "sensitivity_level",
    "sensitivity_rationale", "annotation_confidence", "needs_review",
    "annotation_round", "annotated_at",
)
PREDICTION_COLUMNS = (
    "content_hash", "blind_id", "pred_public_relevance",
    "pred_harm_urgency", "pred_vulnerability", "pred_sensitivity_score",
    "pred_sensitivity_level", "pred_sensitivity_rationale",
    "pred_annotation_confidence", "parse_ok", "parse_error",
    "raw_model_output", "inference_seconds",
)

PROMPT_TEXT = """You are annotating the semantic sensitivity of visible image content.
Judge only what is visibly supported by the image. Do not determine whether the image is real or synthetic. Do not infer the dataset source, provenance, generator, model, prompt, caption, or any unseen context. Do not infer political, medical, conflict, or emergency context unless it is visually supported. Assess the potential sensitivity if imagery with the visible content were deceptive.

Use this frozen rubric:
public_relevance: 0 = private/everyday content; 1 = public event, profession, institution, or recognizable public context; 2 = politics, elections, armed conflict, disaster response, or public health.
harm_urgency: 0 = no plausible immediate public harm; 1 = reputational harm or general public confusion; 2 = physical safety, democratic process, emergency response, or health behavior could be affected.
vulnerability: 0 = no visibly vulnerable/distressed subjects; 1 = patients, displaced people, injured persons, children in crisis, or populations under threat.

Predict only the three component values. Do not calculate or output a total score or sensitivity level. Return exactly one compact JSON object, with no Markdown and no extra keys:
{"public_relevance": 0, "harm_urgency": 0, "vulnerability": 0, "sensitivity_rationale": "short rationale using only visible evidence", "annotation_confidence": "high"}
The component values must be integers in their stated ranges. The rationale must be concise (at most 280 characters). annotation_confidence must be exactly low, medium, or high."""

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
RUN_SESSION_STARTED_PERF = time.perf_counter()

## 3. Utils

Utilities enforce exact JSON keys and ranges, derive score and level in Python, resize only in memory while preserving aspect ratio, and atomically checkpoint every attempted inference.

In [ ]:
@dataclass(frozen=True)
class BlindInferenceItem:
    content_hash: str
    blind_id: str
    image_path: Path


def utc_now() -> str:
    return datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def derived_level(score: int) -> str:
    if score not in range(6):
        raise ValueError(f"sensitivity score outside 0..5: {score}")
    if score <= 1:
        return "low"
    if score <= 3:
        return "medium"
    return "high"


def reject_duplicate_json_keys(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_model_json(raw_output: str) -> dict[str, Any]:
    expected = {
        "public_relevance", "harm_urgency", "vulnerability",
        "sensitivity_rationale", "annotation_confidence",
    }
    payload = json.loads(raw_output.strip(), object_pairs_hook=reject_duplicate_json_keys)
    if not isinstance(payload, dict):
        raise TypeError("response must be one JSON object")
    if set(payload) != expected:
        missing = sorted(expected - set(payload))
        extra = sorted(set(payload) - expected)
        raise ValueError(f"JSON keys do not match schema; missing={missing}, extra={extra}")

    ranges = {
        "public_relevance": range(3),
        "harm_urgency": range(3),
        "vulnerability": range(2),
    }
    for field, allowed in ranges.items():
        value = payload[field]
        if type(value) is not int or value not in allowed:
            raise ValueError(f"{field} must be an integer in {list(allowed)}")

    rationale = payload["sensitivity_rationale"]
    if not isinstance(rationale, str) or not rationale.strip():
        raise ValueError("sensitivity_rationale must be a non-empty string")
    if len(rationale.strip()) > 280:
        raise ValueError("sensitivity_rationale exceeds 280 characters")
    confidence = payload["annotation_confidence"]
    if confidence not in {"low", "medium", "high"}:
        raise ValueError("annotation_confidence must be low, medium, or high")

    payload["sensitivity_rationale"] = rationale.strip()
    score = payload["public_relevance"] + payload["harm_urgency"] + payload["vulnerability"]
    payload["sensitivity_score"] = score
    payload["sensitivity_level"] = derived_level(score)
    return payload


def load_image_in_memory(path: Path) -> Image.Image:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        if max(image.size) > MAX_IMAGE_SIDE:
            image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
        return image.copy()


def atomic_write_csv(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_json(payload: dict[str, Any], path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    os.replace(temporary, path)


def load_prediction_checkpoint(path: Path, allowed_hashes: set[str]) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    frame = pd.read_csv(path, keep_default_na=False)
    if tuple(frame.columns) != PREDICTION_COLUMNS:
        raise RuntimeError("prediction checkpoint schema mismatch")
    if frame["content_hash"].duplicated().any():
        raise RuntimeError("prediction checkpoint contains duplicate content_hash values")
    unexpected = set(frame["content_hash"]) - allowed_hashes
    if unexpected:
        raise RuntimeError(
            f"checkpoint contains hashes outside this Gold run: {sorted(unexpected)[:3]}"
        )
    parse_map = {True: True, False: False, "True": True, "False": False}
    if not frame.empty:
        normalized = frame["parse_ok"].map(parse_map)
        if normalized.isna().any():
            raise RuntimeError("prediction checkpoint contains invalid parse_ok values")
        frame["parse_ok"] = normalized.astype(bool)
    return frame


def json_number(value: Any) -> float | int | None:
    if value is None or pd.isna(value):
        return None
    numeric = float(value)
    return numeric if np.isfinite(numeric) else None

## 4. Data

Validate the frozen Gold file and resolve exactly its 101 hashes through the canonical pilot-600 manifest. Manifest labels and provenance are never selected into the inference queue. The host uses paths only to open images; path strings are never passed to Qwen.

In [ ]:
if not GOLD_CSV_PATH.is_file():
    raise FileNotFoundError(f"Gold annotations not found: {GOLD_CSV_PATH}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"canonical pilot manifest not found: {MANIFEST_PATH}")
if sha256_file(GOLD_CSV_PATH) != EXPECTED_GOLD_SHA256:
    raise RuntimeError("Gold CSV SHA-256 does not match the frozen artifact")

human_gold_locked = pd.read_csv(GOLD_CSV_PATH, keep_default_na=False)
if tuple(human_gold_locked.columns) != GOLD_REQUIRED_COLUMNS:
    raise RuntimeError("Gold CSV columns do not exactly match the required schema")
if len(human_gold_locked) != EXPECTED_GOLD_ROWS:
    raise RuntimeError(f"expected 101 Gold rows, found {len(human_gold_locked)}")
if human_gold_locked["content_hash"].duplicated().any():
    raise RuntimeError("Gold content_hash values are not unique")
if not human_gold_locked["content_hash"].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
    raise RuntimeError("Gold CSV contains an invalid content_hash")

manifest_records: list[dict[str, Any]] = []
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            raise RuntimeError(f"blank manifest line at {line_number}")
        try:
            manifest_records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise RuntimeError(f"invalid manifest JSON at line {line_number}") from exc
if len(manifest_records) != EXPECTED_MANIFEST_ROWS:
    raise RuntimeError(f"expected 600 pilot manifest rows, found {len(manifest_records)}")
manifest_df = pd.DataFrame(manifest_records)
required_manifest_fields = {"content_hash", "relative_image_path"}
if not required_manifest_fields.issubset(manifest_df.columns):
    raise RuntimeError("pilot manifest lacks content_hash or relative_image_path")

gold_hashes = set(human_gold_locked["content_hash"].astype(str))
gold_manifest_rows = manifest_df[manifest_df["content_hash"].isin(gold_hashes)][
    ["content_hash", "relative_image_path"]
].copy()
match_counts = gold_manifest_rows["content_hash"].value_counts()
bad_counts = {
    content_hash: int(match_counts.get(content_hash, 0))
    for content_hash in gold_hashes
    if match_counts.get(content_hash, 0) != 1
}
if bad_counts:
    raise RuntimeError(
        f"Gold hashes must appear exactly once in pilot manifest: {list(bad_counts.items())[:3]}"
    )

gold_identity = human_gold_locked[["content_hash", "blind_id"]].copy()
path_resolution = gold_identity.merge(
    gold_manifest_rows, on="content_hash", how="left", validate="one_to_one"
)
if len(path_resolution) != EXPECTED_GOLD_ROWS:
    raise RuntimeError("did not resolve exactly 101 manifest rows")

pilot_root_resolved = PILOT_ROOT.resolve()
legacy_root_resolved = LEGACY_ROOT.resolve(strict=False)
inference_items_list: list[BlindInferenceItem] = []
for row in path_resolution.itertuples(index=False):
    candidate = (PILOT_ROOT / str(row.relative_image_path)).resolve()
    if not candidate.is_relative_to(pilot_root_resolved):
        raise RuntimeError(f"resolved path escapes canonical pilot root: {candidate}")
    if candidate.is_relative_to(legacy_root_resolved):
        raise RuntimeError(f"legacy image path is forbidden: {candidate}")
    if not candidate.is_file():
        raise FileNotFoundError(f"resolved Gold image does not exist: {candidate}")
    inference_items_list.append(
        BlindInferenceItem(str(row.content_hash), str(row.blind_id), candidate)
    )

inference_items = tuple(inference_items_list)
if len(inference_items) != EXPECTED_GOLD_ROWS:
    raise RuntimeError("expected exactly 101 resolved Gold image paths")
if len({item.image_path for item in inference_items}) != EXPECTED_GOLD_ROWS:
    raise RuntimeError("resolved Gold image paths are not unique")
if any(item.image_path.is_relative_to(legacy_root_resolved) for item in inference_items):
    raise RuntimeError("legacy image tree entered the inference queue")

# Remove manifest-bearing frames before inference. Only opaque IDs and paths remain host-side.
del manifest_df, manifest_records, gold_manifest_rows, path_resolution, inference_items_list
print(
    f"Validated {len(human_gold_locked)} Gold annotations and "
    f"resolved {len(inference_items)} canonical images."
)
print(f"Gold SHA-256: {sha256_file(GOLD_CSV_PATH)}")

## 5. Network

Resolve the current model revision to an immutable Hugging Face commit, then load both processor and unquantized model from that exact SHA. BF16 is selected when supported; otherwise FP16 is used. Loading stops if weights are quantized or offloaded to CPU or disk.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle CUDA GPU accelerator is required for this pilot")

model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION_REQUESTED)
RESOLVED_MODEL_REVISION = model_info.sha
if not RESOLVED_MODEL_REVISION or not re.fullmatch(
    r"[0-9a-f]{40}", RESOLVED_MODEL_REVISION
):
    raise RuntimeError(
        f"Hugging Face did not return an immutable commit SHA: {RESOLVED_MODEL_REVISION!r}"
    )

MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=RESOLVED_MODEL_REVISION,
    trust_remote_code=False,
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=RESOLVED_MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map="auto",
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    trust_remote_code=False,
)
if getattr(model, "is_quantized", False):
    raise RuntimeError("Quantization was not authorized for the initial pilot")
device_map = getattr(model, "hf_device_map", {})
offload_targets = {
    str(device)
    for device in device_map.values()
    if str(device) in {"cpu", "disk"}
}
if offload_targets:
    raise RuntimeError(
        "The unquantized model did not fit fully on Kaggle GPU(s); "
        f"offload targets={sorted(offload_targets)}. Stop here and review memory or "
        "quantization policy rather than silently changing the experiment."
    )
model.eval()

gpu_names = [
    torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())
]
run_metadata: dict[str, Any] = {
    "model_id": MODEL_ID,
    "requested_model_revision": MODEL_REVISION_REQUESTED,
    "resolved_model_revision_sha": RESOLVED_MODEL_REVISION,
    "prompt_text": PROMPT_TEXT,
    "seed": SEED,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "huggingface_hub_version": huggingface_hub.__version__,
    "cuda_version": torch.version.cuda,
    "gpu": gpu_names,
    "dtype": str(MODEL_DTYPE).replace("torch.", ""),
    "generation_parameters": GENERATION_PARAMETERS,
    "image_preprocessing_parameters": {
        "maximum_image_side_pixels": MAX_IMAGE_SIDE,
        "preserve_aspect_ratio": True,
        "resize_location": "memory_only",
        "resampling": "PIL.Image.Resampling.LANCZOS",
        "exif_transpose": True,
        "color_mode": "RGB",
        "processor": "AutoProcessor defaults from the resolved model revision",
    },
    "quantization": None,
}
compatibility_keys = (
    "model_id",
    "resolved_model_revision_sha",
    "prompt_text",
    "seed",
    "dtype",
    "generation_parameters",
    "image_preprocessing_parameters",
)
if PREDICTIONS_PATH.exists() and not RUN_METADATA_PATH.exists():
    raise RuntimeError("prediction checkpoint exists without compatible run metadata")
if RUN_METADATA_PATH.exists():
    prior_metadata = json.loads(RUN_METADATA_PATH.read_text(encoding="utf-8"))
    mismatches = [
        key
        for key in compatibility_keys
        if prior_metadata.get(key) != run_metadata.get(key)
    ]
    if mismatches:
        raise RuntimeError(
            f"existing checkpoint belongs to an incompatible run: {mismatches}"
        )
    run_metadata["started_at_utc"] = prior_metadata.get("started_at_utc", utc_now())
else:
    run_metadata["started_at_utc"] = utc_now()
atomic_write_json(run_metadata, RUN_METADATA_PATH)
print(
    f"Resolved {MODEL_ID}@{RESOLVED_MODEL_REVISION}; "
    f"dtype={MODEL_DTYPE}; GPU={gpu_names}"
)

## 6. Train / Inference

There is no training. This section performs deterministic zero-shot inference under torch.inference_mode(); Qwen weights are never updated. Each model call receives only one in-memory image and the frozen instructions. Human annotations are not accessed or joined here. Parse failures are retained and checkpointed rather than replaced.

In [ ]:
def generate_blind_annotation(image: Image.Image) -> str:
    # Deliberately no hash, blind ID, filename, label, source, provenance, or human data.
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT_TEXT},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs.pop("token_type_ids", None)
    inputs = inputs.to(model.device)
    input_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        generated = model.generate(**inputs, **GENERATION_PARAMETERS)
    generated_only = generated[:, input_length:]
    output = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()
    del inputs, generated, generated_only
    return output


allowed_hashes = {item.content_hash for item in inference_items}
checkpoint = load_prediction_checkpoint(PREDICTIONS_PATH, allowed_hashes)
records_by_hash = {
    str(row["content_hash"]): row.to_dict() for _, row in checkpoint.iterrows()
}

for item in tqdm(
    inference_items, total=EXPECTED_GOLD_ROWS, desc="Gold zero-shot inference"
):
    if item.content_hash in records_by_hash:
        continue

    image = load_image_in_memory(item.image_path)
    torch.cuda.synchronize()
    inference_started = time.perf_counter()
    raw_output = generate_blind_annotation(image)
    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - inference_started

    try:
        parsed = parse_model_json(raw_output)
        parse_ok = True
        parse_error = ""
    except (json.JSONDecodeError, TypeError, ValueError) as exc:
        parsed = None
        parse_ok = False
        parse_error = f"{type(exc).__name__}: {exc}"

    record = {
        "content_hash": item.content_hash,
        "blind_id": item.blind_id,
        "pred_public_relevance": parsed["public_relevance"] if parsed else None,
        "pred_harm_urgency": parsed["harm_urgency"] if parsed else None,
        "pred_vulnerability": parsed["vulnerability"] if parsed else None,
        "pred_sensitivity_score": parsed["sensitivity_score"] if parsed else None,
        "pred_sensitivity_level": parsed["sensitivity_level"] if parsed else None,
        "pred_sensitivity_rationale": (
            parsed["sensitivity_rationale"] if parsed else None
        ),
        "pred_annotation_confidence": (
            parsed["annotation_confidence"] if parsed else None
        ),
        "parse_ok": parse_ok,
        "parse_error": parse_error,
        "raw_model_output": raw_output,
        "inference_seconds": round(inference_seconds, 6),
    }
    records_by_hash[item.content_hash] = record
    ordered_records = [
        records_by_hash[queued.content_hash]
        for queued in inference_items
        if queued.content_hash in records_by_hash
    ]
    atomic_write_csv(
        pd.DataFrame(ordered_records, columns=PREDICTION_COLUMNS), PREDICTIONS_PATH
    )
    del image, raw_output, parsed, record, ordered_records
    gc.collect()

predictions_df = load_prediction_checkpoint(PREDICTIONS_PATH, allowed_hashes)
if len(predictions_df) != EXPECTED_GOLD_ROWS:
    raise RuntimeError("evaluation is blocked until all 101 attempts are checkpointed")
if set(predictions_df["content_hash"]) != allowed_hashes:
    raise RuntimeError("prediction checkpoint does not match the 101 Gold hashes")
prediction_order = {
    item.content_hash: index for index, item in enumerate(inference_items)
}
predictions_df["_order"] = predictions_df["content_hash"].map(prediction_order)
predictions_df = (
    predictions_df.sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)
atomic_write_csv(predictions_df[list(PREDICTION_COLUMNS)], PREDICTIONS_PATH)
print(
    f"Completed attempts: {len(predictions_df)}; "
    f"parse successes: {int(predictions_df['parse_ok'].sum())}"
)

## 7. Evaluation

Only now, after all 101 attempts exist, join predictions to the frozen human annotations. Agreement metrics are computed on strictly parsed predictions, with parse coverage and all-attempt accuracy reported separately. HIGH recall counts parse failures as misses. No arbitrary pass/fail threshold is applied.

In [ ]:
# This is the first prediction and human-annotation join in the notebook.
evaluation_df = predictions_df.merge(
    human_gold_locked,
    on=["content_hash", "blind_id"],
    how="inner",
    validate="one_to_one",
)
if len(evaluation_df) != EXPECTED_GOLD_ROWS:
    raise RuntimeError("prediction and Gold join did not produce exactly 101 rows")

prediction_numeric = [
    "pred_public_relevance",
    "pred_harm_urgency",
    "pred_vulnerability",
    "pred_sensitivity_score",
]
for column in prediction_numeric:
    evaluation_df[column] = pd.to_numeric(evaluation_df[column], errors="coerce")
evaluation_df["inference_seconds"] = pd.to_numeric(
    evaluation_df["inference_seconds"], errors="raise"
)
parsed_eval = evaluation_df[evaluation_df["parse_ok"]].copy()
if parsed_eval.empty:
    raise RuntimeError(
        "No outputs passed strict JSON parsing; agreement metrics cannot be computed"
    )
if parsed_eval[prediction_numeric + ["pred_sensitivity_level"]].isna().any().any():
    raise RuntimeError("a parse_ok row contains missing predicted values")

gold_levels = parsed_eval["sensitivity_level"]
pred_levels = parsed_eval["pred_sensitivity_level"]
gold_scores = parsed_eval["sensitivity_score"].astype(int)
pred_scores = parsed_eval["pred_sensitivity_score"].astype(int)
gold_high_count = int((evaluation_df["sensitivity_level"] == "high").sum())
high_true_positives = int(
    (
        (evaluation_df["sensitivity_level"] == "high")
        & (evaluation_df["pred_sensitivity_level"] == "high")
        & evaluation_df["parse_ok"]
    ).sum()
)

metrics = {
    "attempted_n": len(evaluation_df),
    "parsed_n": len(parsed_eval),
    "parse_success_rate": float(evaluation_df["parse_ok"].mean()),
    "metric_scope": (
        "Agreement metrics use parse_ok rows. Parse failures are separately counted, "
        "are incorrect in all-attempt level accuracy, and are misses for Gold HIGH recall."
    ),
    "sensitivity_level_accuracy": float(
        accuracy_score(gold_levels, pred_levels)
    ),
    "sensitivity_level_accuracy_all_attempts": float(
        (
            evaluation_df["parse_ok"]
            & (
                evaluation_df["pred_sensitivity_level"]
                == evaluation_df["sensitivity_level"]
            )
        ).mean()
    ),
    "sensitivity_level_macro_f1": float(
        f1_score(
            gold_levels,
            pred_levels,
            labels=LEVEL_ORDER,
            average="macro",
            zero_division=0,
        )
    ),
    "sensitivity_level_linear_weighted_kappa": json_number(
        cohen_kappa_score(
            gold_levels, pred_levels, labels=LEVEL_ORDER, weights="linear"
        )
    ),
    "sensitivity_level_quadratic_weighted_kappa": json_number(
        cohen_kappa_score(
            gold_levels, pred_levels, labels=LEVEL_ORDER, weights="quadratic"
        )
    ),
    "exact_sensitivity_score_accuracy": float(
        accuracy_score(gold_scores, pred_scores)
    ),
    "sensitivity_score_mae": float(
        mean_absolute_error(gold_scores, pred_scores)
    ),
    "public_relevance_accuracy": float(
        accuracy_score(
            parsed_eval["public_relevance"],
            parsed_eval["pred_public_relevance"],
        )
    ),
    "harm_urgency_accuracy": float(
        accuracy_score(
            parsed_eval["harm_urgency"], parsed_eval["pred_harm_urgency"]
        )
    ),
    "vulnerability_accuracy": float(
        accuracy_score(
            parsed_eval["vulnerability"], parsed_eval["pred_vulnerability"]
        )
    ),
    "recall_on_human_gold_high": (
        float(high_true_positives / gold_high_count) if gold_high_count else None
    ),
}

gold_distribution = (
    human_gold_locked["sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
)
qwen_distribution = (
    parsed_eval["pred_sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
)
metrics["human_gold_class_distribution"] = gold_distribution.to_dict()
metrics["qwen_parsed_class_distribution"] = qwen_distribution.to_dict()
metrics["parse_failure_count"] = int((~evaluation_df["parse_ok"]).sum())
atomic_write_json(metrics, METRICS_PATH)

evaluation_df["score_error"] = (
    evaluation_df["pred_sensitivity_score"] - evaluation_df["sensitivity_score"]
)
evaluation_df["absolute_score_error"] = evaluation_df["score_error"].abs()
disagreement_mask = (
    ~evaluation_df["parse_ok"]
    | (
        evaluation_df["pred_public_relevance"]
        != evaluation_df["public_relevance"]
    )
    | (evaluation_df["pred_harm_urgency"] != evaluation_df["harm_urgency"])
    | (evaluation_df["pred_vulnerability"] != evaluation_df["vulnerability"])
    | (
        evaluation_df["pred_sensitivity_level"]
        != evaluation_df["sensitivity_level"]
    )
)
disagreements = (
    evaluation_df[disagreement_mask]
    .sort_values(
        ["absolute_score_error", "content_hash"],
        ascending=[False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)
atomic_write_csv(disagreements, DISAGREEMENTS_PATH)

high_gold_misses = evaluation_df[
    (evaluation_df["sensitivity_level"] == "high")
    & evaluation_df["pred_sensitivity_level"].isin(["medium", "low"])
].sort_values(
    ["absolute_score_error", "content_hash"], ascending=[False, True]
)
comparison_distribution = pd.DataFrame(
    {"Human Gold": gold_distribution, "Qwen (parsed)": qwen_distribution}
)

display(pd.Series(metrics, name="value").to_frame())
display(comparison_distribution)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
matrix = confusion_matrix(gold_levels, pred_levels, labels=LEVEL_ORDER)
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=LEVEL_ORDER,
    yticklabels=LEVEL_ORDER,
    ax=axes[0],
)
axes[0].set(
    title="Sensitivity-level confusion matrix (parsed)",
    xlabel="Qwen",
    ylabel="Human Gold",
)
comparison_distribution.plot(
    kind="bar", ax=axes[1], color=["#3b82f6", "#f97316"]
)
axes[1].set(
    title="Gold vs Qwen class distributions",
    xlabel="Sensitivity level",
    ylabel="Count",
)
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print("Disagreements sorted by absolute score error")
display(disagreements)
print("Human Gold HIGH cases predicted medium or low")
display(high_gold_misses)
if (~evaluation_df["parse_ok"]).any():
    print("Strict JSON parse failures")
    display(
        evaluation_df.loc[
            ~evaluation_df["parse_ok"],
            ["content_hash", "blind_id", "parse_error", "raw_model_output"],
        ]
    )

run_metadata.update(
    {
        "completed_at_utc": utc_now(),
        "attempted_images": len(evaluation_df),
        "parsed_images": len(parsed_eval),
        "total_inference_seconds": float(
            evaluation_df["inference_seconds"].sum()
        ),
        "notebook_session_runtime_seconds": float(
            time.perf_counter() - RUN_SESSION_STARTED_PERF
        ),
        "artifacts": {
            "predictions": str(PREDICTIONS_PATH),
            "metrics": str(METRICS_PATH),
            "disagreements": str(DISAGREEMENTS_PATH),
        },
    }
)
atomic_write_json(run_metadata, RUN_METADATA_PATH)
print(f"Artifacts saved under {OUTPUT_DIR}")